## Handling CSV data for our Database Operations


### Creating connection with Database

In [8]:
import oracledb 

username = "system"  
password = "password"  
host = "localhost" 
port = 1521  
service_name = "free" 
connection_string = f"{username}/{password}@{host}:{port}/{service_name}"
connection = oracledb.connect(connection_string)
print("Connection established.")

Connection established.


### Creating Data Table with the same Schema as our Actual CSV file

In [9]:
with connection.cursor() as cursor:
    try:
        cursor.execute("drop table t")
    except oracledb.DatabaseError:
        pass

    cursor.execute("""create table t (k number, 
                                      first_name varchar2(30), 
                                      last_name varchar2(30), 
                                      country varchar2(30))""")

#### Checking the requirements for CSV format

In [10]:
import csv 

In [ ]:
batch_size = 10000 ## Batch size for executemany() - adjust as needed based on memory and performance requirements

with connection.cursor() as cursor:
    
    sql = "insert into t (k, first_name, last_name, country) values (:1, :2, :3, :4)"
    
    # Predefine memory areas to match the table definition (or max data) to avoid memory re-allocs
    cursor.setinputsizes(None, 30, 30, 30)

    with open("csv/data1.csv", "r") as csv_file:
        csv_reader = csv.reader(csv_file, delimiter=',') ### .reader method returns an iterable that produces each row as a list of strings
        data = []
        for line in csv_reader:
            data.append((line[0], line[1], line[2], line[3]))   # e.g [('1', 'Fred', 'Nurke', 'UK')]
            if len(data) % batch_size == 0:
                cursor.executemany(sql, data)
                data = []
        if data:
            cursor.executemany(sql, data)
        connection.commit()

print("Done")

Done


### Print the actual saved data from the Database 

In [12]:
with connection.cursor() as cursor:
    sql = "select * from t order by k"
    for r in cursor.execute(sql):
        print(r)

(1, 'Fred', 'Nurke', 'UK')
(2, 'Henry', 'Crun', 'UK')
(3, 'Gourav', 'Yadav', 'IN')
(4, 'John', 'Smith', 'US')
(5, 'Jane', 'Doe', 'US')
(6, 'Emily', 'Davis', 'CA')
(7, 'Michael', 'Brown', 'AU')
(8, 'Sarah', 'Wilson', 'UK')
(9, 'David', 'Johnson', 'US')
(10, 'Laura', 'Garcia', 'ES')
(11, 'James', 'Miller', 'US')
(12, 'Olivia', 'Martinez', 'MX')
(13, 'Robert', 'Anderson', 'US')
(14, 'Jessica', 'Lee', 'CA')
(15, 'William', 'Clark', 'UK')
(16, 'Emily', 'Walker', 'AU')
(17, 'Daniel', 'Hall', 'US')
(18, 'Sophia', 'Young', 'UK')
(19, 'Matthew', 'Allen', 'US')
(20, 'Isabella', 'King', 'AU')


## Wrinting CSV file from the Database Table 

In [15]:
import csv
import time


In [ ]:
sql = "select * from T order by k" ## SQL query to select all rows from table T ordered by column k

with connection.cursor() as cursor:
    start = time.time()
    cursor.arraysize = 1000
    with open("testwrite.csv", "w", encoding="utf-8") as outputfile:
        writer = csv.writer(outputfile, lineterminator="\n") ## Writer method returns a writer object that converts the user's data into delimited strings on the given file-like object. lineterminator is set to "\n" to ensure consistent line endings across platforms.
        results = cursor.execute(sql)
        writer.writerows(results) ## writerows() method takes an iterable of rows (like the cursor results) and writes them to the CSV file in one call, which is more efficient than writing each row individually.

    elapsed = time.time() - start
    print("Writing CSV: 10000 rows in {:06.4f} seconds".format(elapsed))  

Writing CSV: 10000 rows in 0.0176 seconds
